# 🛡️ Network Intrusion Detection System — ML Training Pipeline

This notebook trains a **Random Forest** classifier on the **CICIDS2017** dataset to distinguish benign network traffic from attacks. The trained model and scaler are exported for use in the live Streamlit dashboard.

---

## Phase 0: Environment Setup and Imports

### Step 1: Import Libraries

Import all required libraries: `pandas`, `numpy`, `scikit-learn`, and `joblib`.

In [1]:
import os
import glob
import warnings

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import joblib

warnings.filterwarnings('ignore')

print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"joblib:       {joblib.__version__}")
print("\n✅ All libraries loaded successfully.")

pandas:       2.3.3
numpy:        2.4.2
scikit-learn: 1.8.0
joblib:       1.5.3

✅ All libraries loaded successfully.


### ⏸️ Checkpoint 0

**Wait for confirmation** that all libraries loaded without errors before proceeding.

---
## Phase 1: Data Acquisition and Feature Reduction

### Step 1: Load the CICIDS2017 Dataset

Load **all CSV files** from the `data/` directory and concatenate them into a single DataFrame. The dataset ships as multiple day-files.

In [2]:
DATA_DIR = os.path.join('..', 'data')

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  • {os.path.basename(f)}")

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in '{os.path.abspath(DATA_DIR)}'.\n"
        "Download CICIDS2017 MachineLearningCSV.zip and extract into the data/ folder."
    )

# Load and concatenate all CSVs
dfs = []
for f in csv_files:
    print(f"Loading {os.path.basename(f)}...", end=" ")
    chunk = pd.read_csv(f, encoding='utf-8', low_memory=False)
    print(f"({len(chunk):,} rows)")
    dfs.append(chunk)

df_raw = pd.concat(dfs, ignore_index=True)

# CRITICAL: Strip whitespace from column names (CICIDS2017 has leading spaces)
df_raw.columns = df_raw.columns.str.strip()

print(f"\n📊 Total dataset shape: {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns[:10])}... ({len(df_raw.columns)} total)")

Found 8 CSV files:
  • Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  • Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  • Friday-WorkingHours-Morning.pcap_ISCX.csv
  • Monday-WorkingHours.pcap_ISCX.csv
  • Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  • Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  • Tuesday-WorkingHours.pcap_ISCX.csv
  • Wednesday-workingHours.pcap_ISCX.csv
Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv... (225,745 rows)
Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv... (286,467 rows)
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv... (191,033 rows)
Loading Monday-WorkingHours.pcap_ISCX.csv... (529,918 rows)
Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv... (288,602 rows)
Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv... (170,366 rows)
Loading Tuesday-WorkingHours.pcap_ISCX.csv... (445,909 rows)
Loading Wednesday-workingHours.pcap_ISCX.csv... (692,703 rows)

📊 Total da

### Step 2: 🔑 Feature Reduction (CRITICAL)

Select **only** the 7 features that can be realistically computed from raw packets in real-time:

| # | Feature | Why it's extractable in real-time |
|---|---------|-----------------------------------|
| 1 | `Destination Port` | Directly from TCP header |
| 2 | `Total Fwd Packets` | Count packets src→dst |
| 3 | `Total Backward Packets` | Count packets dst→src |
| 4 | `Fwd Packet Length Mean` | Mean payload size (forward) |
| 5 | `Bwd Packet Length Mean` | Mean payload size (backward) |
| 6 | `SYN Flag Count` | Count SYN flags in TCP header |
| 7 | `ACK Flag Count` | Count ACK flags in TCP header |

In [3]:
SELECTED_FEATURES = [
    'Destination Port',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'SYN Flag Count',
    'ACK Flag Count',
]

LABEL_COL = 'Label'

# Verify all columns exist
all_needed = SELECTED_FEATURES + [LABEL_COL]
missing = [c for c in all_needed if c not in df_raw.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}. Available: {list(df_raw.columns)}")

df = df_raw[all_needed].copy()
print(f"✅ Reduced dataset shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

✅ Reduced dataset shape: (2830743, 8)
   Columns: ['Destination Port', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'SYN Flag Count', 'ACK Flag Count', 'Label']


### Step 3: Data Cleaning and Label Encoding

- Drop rows with `NaN` or `Infinity` values.
- Encode the `Label` column: `BENIGN` → **0**, everything else → **1** (Attack).

In [4]:
# Replace infinities with NaN, then drop all NaN rows
df.replace([np.inf, -np.inf], np.nan, inplace=True)
rows_before = len(df)
df.dropna(inplace=True)
rows_after = len(df)
print(f"Dropped {rows_before - rows_after:,} rows with NaN/Inf values.")
print(f"Remaining rows: {rows_after:,}")

# Show unique labels before encoding
print(f"\nUnique labels ({df[LABEL_COL].nunique()}):")
print(df[LABEL_COL].value_counts().to_string())

# Encode: BENIGN=0, everything else=1
df[LABEL_COL] = df[LABEL_COL].apply(lambda x: 0 if x.strip() == 'BENIGN' else 1)

print(f"\n📊 Encoded class distribution:")
print(f"   Benign (0): {(df[LABEL_COL] == 0).sum():>10,}")
print(f"   Attack (1): {(df[LABEL_COL] == 1).sum():>10,}")
print(f"   Total:      {len(df):>10,}")

Dropped 0 rows with NaN/Inf values.
Remaining rows: 2,830,743

Unique labels (15):
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11

📊 Encoded class distribution:
   Benign (0):  2,273,097
   Attack (1):    557,646
   Total:       2,830,743


### ⏸️ Checkpoint 1

**Verify the output above:**
- The reduced dataset should have **8 columns** (7 features + 1 label).
- Class distribution should show both Benign and Attack samples.

**Wait for confirmation** before proceeding to model training.

---
## Phase 2: Model Training and Export

### Step 1: Feature Scaling

Fit a `StandardScaler` on the features and **save it** — the live dashboard needs the exact same scaler.

In [5]:
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

X = df[SELECTED_FEATURES].values
y = df[LABEL_COL].values

print(f"Features shape: {X.shape}")
print(f"Labels shape:   {y.shape}")

# Fit and save the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaler_path = os.path.join(MODELS_DIR, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler saved to: {os.path.abspath(scaler_path)}")

Features shape: (2830743, 7)
Labels shape:   (2830743,)

💾 Scaler saved to: c:\Vault\Projects\network-security-project\models\scaler.joblib


### Step 2: Train/Test Split and Model Training

80/20 stratified split, then train a `RandomForestClassifier` with 100 trees.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"\nTraining the Random Forest (100 trees)...")

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
model.fit(X_train, y_train)

print("\n✅ Training complete!")

Training set: 2,264,594 samples
Test set:     566,149 samples

Training the Random Forest (100 trees)...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=-1)]: Done   6 tasks      | elapsed:    4.2s



✅ Training complete!


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   20.0s finished


### Step 3: Model Evaluation

Print the **F1-score** (weighted) and **Confusion Matrix**.

In [7]:
y_pred = model.predict(X_test)

f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred)

print("=" * 50)
print(f"  Weighted F1-Score: {f1:.4f}")
print("=" * 50)

print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                 Benign  Attack")
print(f"  Actual Benign  {cm[0][0]:>7,}  {cm[0][1]:>6,}")
print(f"  Actual Attack  {cm[1][0]:>7,}  {cm[1][1]:>6,}")

print(f"\n📋 Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

[Parallel(n_jobs=22)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=22)]: Done   6 tasks      | elapsed:    0.0s


  Weighted F1-Score: 0.9905

Confusion Matrix:
                 Predicted
                 Benign  Attack
  Actual Benign  450,948   3,672
  Actual Attack    1,704  109,825

📋 Full Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.99      0.99    454620
      Attack       0.97      0.98      0.98    111529

    accuracy                           0.99    566149
   macro avg       0.98      0.99      0.99    566149
weighted avg       0.99      0.99      0.99    566149



[Parallel(n_jobs=22)]: Done 100 out of 100 | elapsed:    0.2s finished


### Step 4: Export the Trained Model

Save the trained Random Forest model to a `.joblib` file in the `models/` directory.

In [8]:
model_path = os.path.join(MODELS_DIR, 'rf_model.joblib')
joblib.dump(model, model_path)

print(f"💾 Model saved to: {os.path.abspath(model_path)}")
print(f"   Model file size: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

# Verify both files exist
print(f"\n📁 Files in models/ directory:")
for f in os.listdir(MODELS_DIR):
    fpath = os.path.join(MODELS_DIR, f)
    print(f"   {f} ({os.path.getsize(fpath) / 1024 / 1024:.1f} MB)")

print("\n🎉 Pipeline complete! You can now run the live dashboard.")

💾 Model saved to: c:\Vault\Projects\network-security-project\models\rf_model.joblib
   Model file size: 30.7 MB

📁 Files in models/ directory:
   rf_model.joblib (30.7 MB)
   scaler.joblib (0.0 MB)

🎉 Pipeline complete! You can now run the live dashboard.


### ⏸️ Checkpoint 2

**Review before moving on:**
- F1-score should be high (typically > 0.95 for this dataset and these features).
- `models/rf_model.joblib` and `models/scaler.joblib` should both exist.

**Wait for confirmation** before proceeding to build the live Streamlit application.